In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)  # 601 = AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]

rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]
df.head()

,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))

Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


In [6]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "126ac41d-e5e6-4702-b949-9d141a1b7042"
RESOURCE_GROUP = "rg-mlab1-ali886"
WORKSPACE = "mlw-lab1-ali886"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: mlw-lab1-ali886


In [7]:
import os, shutil
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder = "../data/mltable_ai4i"
os.makedirs(folder, exist_ok=True)
shutil.copy("../data/ai4i_clean.csv", f"{folder}/ai4i_clean.csv")

with open(f"{folder}/MLTable", "w") as f:
    f.write("""type: mltable
paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
""")

table_asset = Data(
    name="ai4i-maintenance-table",
    version="2",
    type=AssetTypes.MLTABLE,
    path=folder,
    description="AI4I 2020 predictive maintenance, cleaned (mltable for AutoML)",
)
table_asset = ml_client.data.create_or_update(table_asset)
print("Registered:", table_asset.name, "version", table_asset.version)

HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "835720a8e082847aa4c676516aea063f",
        "request": "f56fc3d27a9ead7f"
    }
}Type: Environment
Info: {
    "value": "uaenorth"
}Type: Location
Info: {
    "value": "uaenorth"
}Type: Time
Info: {
   

In [ ]:
import pandas as pd

df = pd.read_csv("../data/ai4i_clean.csv")
df["type"] = df["type"].map({"L": 0, "M": 1, "H": 2})
df["machine_failure"] = df["machine_failure"].astype(int)
df.to_csv("../data/ai4i_automl.csv", index=False)
print(df.dtypes)
print(df.head())

In [8]:
import os, shutil
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

ml_client = MLClient(DefaultAzureCredential(), "126ac41d-e5e6-4702-b949-9d141a1b7042",
                     "rg-mlab1-ali886", "mlw-lab1-ali886")

os.makedirs("../data/mltable_numeric", exist_ok=True)
shutil.copy("../data/ai4i_automl.csv", "../data/mltable_numeric/ai4i_automl.csv")
with open("../data/mltable_numeric/MLTable", "w") as f:
    f.write("""paths:
  - file: ./ai4i_automl.csv
transformations:
  - read_delimited:
      delimiter: ','
      header: all_files_same_headers
      encoding: utf8
""")

asset = Data(
    name="ai4i-maintenance-table-numeric",
    version="1",
    type=AssetTypes.MLTABLE,
    path="../data/mltable_numeric",
    description="AI4I numeric-encoded for AutoML KNN",
)
ml_client.data.create_or_update(asset)
print("Registered")

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "3c23d8e6f2603d49a06a45e7a44e9ef2",
        "request": "c47c49141580e629"
    }
}Type: Environment
Info: {
    "value": "uaenorth"
}Type: Location
Info: {
    "value": "uaenorth"
}Type: Time
Info: {
   